# Phase 5 — Heavy Tails: Exploration Notebook

This notebook explores:
- Tail behaviour of different distributions
- Hill estimator for tail index
- Budget impact of distribution choice (VaR, ES)
- The practical cost of assuming Normality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_gen import generate_severance_data
from src.heavy_tails import hill_estimator, hill_plot_data, tail_probability, compare_tail_risk
from src.budget_impact import (
    var_at_level, expected_shortfall, budget_reserve,
    analytical_var_pareto, analytical_var_normal,
    analytical_es_pareto, analytical_es_normal,
    compare_distributions_impact,
)
from src.fitting import fit_all

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. Tail Probability Comparison

In [ ]:
# Pareto severance model
alpha, x_m = 2.5, 10000.0
mu = alpha * x_m / (alpha - 1)
sigma = np.sqrt(alpha * x_m**2 / ((alpha-1)**2 * (alpha-2)))

distributions = {
    "pareto:Pareto(2.5)": {"alpha": alpha, "x_m": x_m},
    "normal:Normal (matched)": {"mu": mu, "sigma": sigma},
}

thresholds = [20000, 30000, 50000, 75000, 100000, 150000]
df = compare_tail_risk(thresholds, distributions)
df

## 2. Hill Estimator on Severance Data

In [ ]:
severance = generate_severance_data(2000, seed=SEED)

k_values, alpha_estimates = hill_plot_data(severance)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, alpha_estimates, "o-", markersize=4, color="#E63946")
ax.axhline(2.5, color="green", linestyle="--", label="True alpha = 2.5")
ax.set_xlabel("k (number of upper order statistics)")
ax.set_ylabel("Hill estimate of alpha")
ax.set_title("Hill Plot: Tail Index Estimation")
ax.legend()
ax.set_ylim(0, 6)
plt.tight_layout()
plt.show()

# Point estimates at different k
for k in [20, 50, 100, 200]:
    alpha_hat = hill_estimator(severance, k)
    print(f"k={k:>4}: alpha_hat = {alpha_hat:.3f}")

## 3. VaR and ES Comparison

In [ ]:
print("Budget Risk Metrics: Pareto vs Normal")
print("=" * 55)
print(f"{'Metric':<20} {'Pareto':>12} {'Normal':>12} {'Ratio':>10}")
print("-" * 55)

for cl in [0.90, 0.95, 0.99]:
    var_p = analytical_var_pareto(alpha, x_m, cl)
    var_n = analytical_var_normal(mu, sigma, cl)
    print(f"VaR {int(cl*100)}%{'':>12} R${var_p:>9,.0f} R${var_n:>9,.0f} {var_p/var_n:>9.2f}x")

print()
for cl in [0.90, 0.95, 0.99]:
    es_p = analytical_es_pareto(alpha, x_m, cl)
    es_n = analytical_es_normal(mu, sigma, cl)
    print(f"ES {int(cl*100)}%{'':>13} R${es_p:>9,.0f} R${es_n:>9,.0f} {es_p/es_n:>9.2f}x")

## 4. Full Budget Impact Comparison

In [ ]:
# Fit multiple distributions to severance data
results = fit_all(severance, candidates=["normal", "lognormal", "gamma", "pareto"])

impact_df = compare_distributions_impact(results, n_simulations=100_000, seed=SEED)
impact_df

## 5. Visual: Budget Reserve by Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

dists = impact_df["Distribution"]
reserve_95 = impact_df["Reserve_95%"]
reserve_99 = impact_df["Reserve_99%"]

x_pos = np.arange(len(dists))
width = 0.35

ax.bar(x_pos - width/2, reserve_95, width, label="Reserve at 95%", color="#457B9D")
ax.bar(x_pos + width/2, reserve_99, width, label="Reserve at 99%", color="#E63946")
ax.set_xticks(x_pos)
ax.set_xticklabels(dists)
ax.set_ylabel("Budget Reserve Above Mean (R$)")
ax.set_title("Budget Reserve Needed by Distribution Choice")
ax.legend()
plt.tight_layout()
plt.show()